# LINK drill — answers

Open only when marking.


## L1


In [ ]:
import csv

def LoadBays(filename):
    clean, rejected = [], []
    with open(filename, encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            if len(row) < 4:
                rejected.append("short row")
                continue
            code, vessel, berth, tonnes = [x.strip() for x in row]
            if code == "" or vessel == "":
                rejected.append("empty code or vessel")
                continue
            if not tonnes.lstrip("-").isdigit():
                rejected.append("tonnes not int")
                continue
            clean.append((code, vessel, berth, int(tonnes)))
    return clean, rejected

clean, rejected = LoadBays("LINK_BAYS.csv")
print(len(clean), len(rejected), clean[0])


## L2


In [ ]:
def InsertionSort(records):
    arr = records[:]
    for i in range(1, len(arr)):
        current = arr[i]
        j = i - 1
        while j >= 0 and (arr[j][0], arr[j][1]) > (current[0], current[1]):
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = current
    return arr

def FindFirst(records, code):
    low, high = 0, len(records) - 1
    found = -1
    while low <= high:
        middle = (low + high) // 2
        if records[middle][0] == code:
            found = middle
            high = middle - 1
        elif records[middle][0] < code:
            low = middle + 1
        else:
            high = middle - 1
    return found

def ListVessels(records, code):
    index = FindFirst(records, code)
    if index == -1:
        return []
    out = []
    while index < len(records) and records[index][0] == code:
        out.append(records[index][1])
        index += 1
    return out

ordered = InsertionSort(clean)
print(FindFirst(ordered, "B07"), FindFirst(ordered, "Z99"))
print(ListVessels(ordered, "B07"))
print(ListVessels(ordered, "Z99"))


## L3


In [ ]:
class Call:
    def __init__(self, ref, vessel):
        self.__ref = ref
        self.__vessel = vessel
        self.__next = None

    def getRef(self):
        return self.__ref

    def getVessel(self):
        return self.__vessel

    def getNext(self):
        return self.__next

    def setNext(self, nxt):
        self.__next = nxt

class Lane:
    def __init__(self):
        self.__first = None

    def getFirst(self):
        return self.__first

    def AddCall(self, new_call):
        if self.__first is None:
            self.__first = new_call
            return True
        if new_call.getRef() == self.__first.getRef():
            print("Duplicate ref")
            return False
        if new_call.getRef() < self.__first.getRef():
            new_call.setNext(self.__first)
            self.__first = new_call
            return True
        current = self.__first
        while current.getNext() is not None and current.getNext().getRef() < new_call.getRef():
            current = current.getNext()
        nxt = current.getNext()
        if nxt is not None and nxt.getRef() == new_call.getRef():
            print("Duplicate ref")
            return False
        new_call.setNext(nxt)
        current.setNext(new_call)
        return True

    def Brief(self):
        out = []
        current = self.__first
        while current is not None:
            out.append((current.getRef(), current.getVessel()))
            current = current.getNext()
        return out

bay_lane = Lane()
bay_lane.AddCall(Call(30, "South Wind"))
bay_lane.AddCall(Call(10, "Amber"))
bay_lane.AddCall(Call(20, "Coral"))
bay_lane.AddCall(Call(10, "Amber"))
print(bay_lane.Brief())


## L4


In [ ]:
from pathlib import Path
from flask import Flask, render_template, request, redirect, url_for

link_dir = Path("templates/link")
link_dir.mkdir(parents=True, exist_ok=True)
(link_dir / "desk.html").write_text(
    "<!doctype html><html><body><h1>bay desk</h1>"
    "{% if error %}<p>{{ error }}</p>{% endif %}"
    "<form method=\"POST\" action=\"{{ url_for('desk') }}\">"
    "<input type=\"text\" name=\"berth_code\">"
    "<input type=\"submit\" value=\"open\"></form></body></html>",
    encoding="utf-8",
)
(link_dir / "lane.html").write_text(
    "<!doctype html><html><body><h1>{{ berth_code }}</h1><ul>"
    "{% for row in rows %}<li>{{ row[1] }} — berth {{ row[2] }}</li>"
    "{% else %}<li>none</li>{% endfor %}</ul>"
    "<p><a href=\"{{ url_for('desk') }}\">back</a></p></body></html>",
    encoding="utf-8",
)

app = Flask(__name__)

@app.route("/", methods=["GET", "POST"])
def desk():
    error = None
    if request.method == "POST":
        berth_code = request.form.get("berth_code", "").strip()
        if berth_code:
            return redirect(url_for("lane", berth_code=berth_code))
        error = "Please fill in all fields"
    return render_template("link/desk.html", error=error)

@app.route("/lane/<berth_code>")
def lane(berth_code):
    rows = [row for row in ordered if str(row[2]) == str(berth_code)]
    return render_template("link/lane.html", rows=rows, berth_code=berth_code)

lane_rows = [row for row in ordered if str(row[2]) == "1"]
print(lane_rows)
